In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [2]:
import json, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "corpus").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "scripts"))
from chunking import chunk_words
from golden import facts, load_golden, norm, recall_at_k

docs = {p.stem: p.read_text(encoding="utf-8") for p in sorted((ROOT / "corpus/text").glob("*.txt"))}
golden = load_golden(ROOT / "eval/golden.jsonl")
print(len(docs), "documents,", len(golden), "questions")

5 documents, 30 questions


In [3]:
import pandas as pd

fact_list = [(g["id"], alts) for g in golden for alts in facts(g)]
CONFIGS = [(64, 0), (64, 32), (128, 0), (128, 32), (192, 0), (192, 48)]

def build_chunks(size, overlap):
    # chunk each document separately so no chunk mixes two documents
    return [{"doc": name, "text": c} for name, t in docs.items() for c in chunk_words(t, size, overlap)]

rows = []
for size, ov in CONFIGS:
    texts = [norm(c["text"]) for c in build_chunks(size, ov)]
    covered = sum(any(norm(a) in t for a in alts for t in texts) for _, alts in fact_list)
    rows.append({"size": size, "overlap": ov, "chunks": len(texts), "facts_covered": f"{covered}/{len(fact_list)}"})
pd.DataFrame(rows)

,size,overlap,chunks,facts_covered
0,64,0,341,23/32
1,64,32,676,32/32
2,128,0,171,27/32
3,128,32,226,32/32
4,192,0,116,29/32
5,192,48,153,32/32


In [4]:
from transformers import AutoTokenizer

MODELS = {
    "e5": ("intfloat/multilingual-e5-base", "passage: "),
    "tr": ("trmteb/turkish-embedding-model", ""),
}
toks = {k: AutoTokenizer.from_pretrained(name) for k, (name, _) in MODELS.items()}

rows = []
for size, ov in CONFIGS:
    chunks = build_chunks(size, ov)
    row = {"size": size, "overlap": ov}
    for k, (_, prefix) in MODELS.items():
        lens = [len(toks[k](prefix + c["text"])["input_ids"]) for c in chunks]
        row[f"{k}_max"] = max(lens)
        row[f"{k}_over_512"] = sum(l > 512 for l in lens)
    rows.append(row)
pd.DataFrame(rows)

,size,overlap,e5_max,e5_over_512,tr_max,tr_over_512
0,64,0,150,0,150,0
1,64,32,150,0,158,0
2,128,0,294,0,293,0
3,128,32,294,0,293,0
4,192,0,418,0,405,0
5,192,48,412,0,400,0


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

RUN_CONFIGS = [(64, 0), (64, 32), (128, 32), (192, 48)]
QUERY_PREFIX = {"e5": "query: ", "tr": ""}
models = {k: SentenceTransformer(name) for k, (name, _) in MODELS.items()}

def embed(k, texts, is_query):
    prefix = QUERY_PREFIX[k] if is_query else MODELS[k][1]
    return models[k].encode([prefix + t for t in texts], normalize_embeddings=True,
                            batch_size=32, show_progress_bar=False)

# rankings[(model, size, overlap)][question_id] = chunk texts sorted from most to least similar
rankings = {}
for size, ov in RUN_CONFIGS:
    chunks = build_chunks(size, ov)
    texts = [c["text"] for c in chunks]
    for k in MODELS:
        chunk_vecs = embed(k, texts, is_query=False)
        q_vecs = embed(k, [g["question"] for g in golden], is_query=True)
        scores = q_vecs @ chunk_vecs.T          # cosine similarity, vectors are normalized
        order = np.argsort(-scores, axis=1)[:, :10]
        rankings[(k, size, ov)] = {g["id"]: [texts[i] for i in order[n]] for n, g in enumerate(golden)}
        print(f"done: {k} size={size} overlap={ov} chunks={len(texts)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

done: e5 size=64 overlap=0 chunks=341


done: tr size=64 overlap=0 chunks=341


done: e5 size=64 overlap=32 chunks=676


done: tr size=64 overlap=32 chunks=676


done: e5 size=128 overlap=32 chunks=226


done: tr size=128 overlap=32 chunks=226


done: e5 size=192 overlap=48 chunks=153


done: tr size=192 overlap=48 chunks=153


In [6]:
# BM25 retrieval, same chunks as the embedding runs
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "corpus").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from bm25 import BM25
from textproc import tokenize
from chunking import chunk_words
from golden import load_golden, facts, recall_at_k

texts = {p.stem: p.read_text(encoding="utf-8") for p in sorted((ROOT / "corpus/text").glob("*.txt"))}
gold = load_golden(ROOT / "eval/golden.jsonl")
if "rankings" not in globals():
    rankings = {}

for size, ov in [(64, 0), (64, 32), (128, 32), (192, 48)]:
    chunks = [c for t in texts.values() for c in chunk_words(t, size, ov)]
    for stem in ["none", "prefix5"]:
        bm = BM25([tokenize(c, stem) for c in chunks])
        name = f"bm25-{stem}"
        rankings[(name, size, ov)] = {
            g["id"]: [chunks[i] for i, _ in bm.search(tokenize(g["question"], stem), k=10)]
            for g in gold
        }
        print(f"done: {name} size={size} overlap={ov} chunks={len(chunks)}")

rows = []
for (model, size, ov), ranked in rankings.items():
    r = {"model": model, "size": size, "overlap": ov}
    for k in (1, 3, 5, 10):
        r[f"R@{k}"] = round(sum(recall_at_k(ranked[g["id"]], facts(g), k) for g in gold) / len(gold), 2)
    rows.append(r)
summary = pd.DataFrame(rows).sort_values(["size", "overlap", "model"]).reset_index(drop=True)
summary

done: bm25-none size=64 overlap=0 chunks=341
done: bm25-prefix5 size=64 overlap=0 chunks=341
done: bm25-none size=64 overlap=32 chunks=676


done: bm25-prefix5 size=64 overlap=32 chunks=676
done: bm25-none size=128 overlap=32 chunks=226
done: bm25-prefix5 size=128 overlap=32 chunks=226
done: bm25-none size=192 overlap=48 chunks=153
done: bm25-prefix5 size=192 overlap=48 chunks=153


,model,size,overlap,R@1,R@3,R@5,R@10
0,bm25-none,64,0,0.07,0.13,0.17,0.20
1,bm25-prefix5,64,0,0.27,0.30,0.33,0.37
2,e5,64,0,0.30,0.30,0.32,0.45
3,tr,64,0,0.10,0.20,0.33,0.43
4,bm25-none,64,32,0.07,0.10,0.13,0.17
5,bm25-prefix5,64,32,0.20,0.30,0.30,0.37
6,e5,64,32,0.27,0.37,0.40,0.43
7,tr,64,32,0.13,0.17,0.27,0.53
8,bm25-none,128,32,0.07,0.13,0.17,0.20
9,bm25-prefix5,128,32,0.13,0.30,0.30,0.50


In [7]:
# Hybrid retrieval: fuse the embedding and BM25 rankings with Reciprocal Rank Fusion
from fusion import rrf

HYBRIDS = {
    "hyb-e5+bm25": ["e5", "bm25-prefix5"],
    "hyb-tr+bm25": ["tr", "bm25-prefix5"],
    "hyb-e5+tr+bm25": ["e5", "tr", "bm25-prefix5"],
}

for size, ov in RUN_CONFIGS:
    for name, parts in HYBRIDS.items():
        if all((p, size, ov) in rankings for p in parts):
            rankings[(name, size, ov)] = {
                g["id"]: rrf([rankings[(p, size, ov)][g["id"]] for p in parts])
                for g in golden
            }
            print(f"done: {name} size={size} overlap={ov}")

done: hyb-e5+bm25 size=64 overlap=0
done: hyb-tr+bm25 size=64 overlap=0
done: hyb-e5+tr+bm25 size=64 overlap=0
done: hyb-e5+bm25 size=64 overlap=32
done: hyb-tr+bm25 size=64 overlap=32
done: hyb-e5+tr+bm25 size=64 overlap=32
done: hyb-e5+bm25 size=128 overlap=32
done: hyb-tr+bm25 size=128 overlap=32
done: hyb-e5+tr+bm25 size=128 overlap=32
done: hyb-e5+bm25 size=192 overlap=48
done: hyb-tr+bm25 size=192 overlap=48
done: hyb-e5+tr+bm25 size=192 overlap=48


In [8]:
rows = []
for (k, size, ov), ranks in rankings.items():
    row = {"model": k, "size": size, "overlap": ov}
    for K in (1, 3, 5, 10):
        scores = []
        for g in golden:
            scores.append(recall_at_k(ranks[g["id"]], facts(g), K))
        row[f"R@{K}"] = np.mean(scores)
    rows.append(row)
pd.DataFrame(rows).round(2)

,model,size,overlap,R@1,R@3,R@5,R@10
0,e5,64,0,0.30,0.30,0.32,0.45
1,tr,64,0,0.10,0.20,0.33,0.43
2,e5,64,32,0.27,0.37,0.40,0.43
3,tr,64,32,0.13,0.17,0.27,0.53
4,e5,128,32,0.27,0.43,0.47,0.58
5,tr,128,32,0.20,0.40,0.50,0.57
6,e5,192,48,0.22,0.48,0.52,0.72
7,tr,192,48,0.17,0.43,0.53,0.65
8,bm25-none,64,0,0.07,0.13,0.17,0.20
9,bm25-prefix5,64,0,0.27,0.30,0.33,0.37


In [9]:
detail = {}
for (k, size, ov), ranks in rankings.items():
    col = f"{k}-{size}/{ov}"
    detail[col] = {
        g["id"]: recall_at_k(ranks[g["id"]], facts(g), 10)
        for g in golden
    }
d = pd.DataFrame(detail)
d.insert(0, "type", [g["type"] for g in golden])
d.insert(1, "question", [g["question"][:45] for g in golden])
d

,type,question,e5-64/0,tr-64/0,e5-64/32,tr-64/32,e5-128/32,tr-128/32,e5-192/48,tr-192/48,...,hyb-e5+tr+bm25-64/0,hyb-e5+bm25-64/32,hyb-tr+bm25-64/32,hyb-e5+tr+bm25-64/32,hyb-e5+bm25-128/32,hyb-tr+bm25-128/32,hyb-e5+tr+bm25-128/32,hyb-e5+bm25-192/48,hyb-tr+bm25-192/48,hyb-e5+tr+bm25-192/48
q001,paraphrase,Kişilerin verileri hangi durumlarda temizlenm,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
q002,dogrudan,Açık rıza nedir?,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
q003,dogrudan,Müşteri hakkında hangi verileri tuttuğumuzu s,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,...,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
q004,senaryo,"Sitemiz hacklendi, müşteri bilgileri sızdı. K",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
q005,paraphrase,Silinmesi gereken verileri en geç kaç ayda bi,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
q006,paraphrase,Kullanıcı sormasa da verileri ne için topladı,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
q007,senaryo,KVKK onayıyla 'metni okudum' kutusunu tek che,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
q008,senaryo,Şifreleme key'ini .env'de tutmak sorun olur m,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
q009,paraphrase,Klavyede yazma ritmi hassas veri sayılır mı?,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
q010,coklu,Müşteri verisini silmemi istedi ama kargo fir,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,...,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0


In [10]:
def show(qid, key=("e5", 192, 48), n=3):
    for i, t in enumerate(rankings[key][qid][:n], 1):
        print(i, t[:250], "\n")

show("q008")

1 alınarak gönderilmesi gerekmektedir. Ayrıca çalışanların şahsi elektronik cihazlarının, bilgi sistem ağına erişim sağlaması da güvenlik ihlali riskini arttırdığından bunlar için de mutlaka yeterli güvenlik tedbirleri alınmalıdır. Kişisel veri güvenli 

2 muhafaza edilmesi, b) Kriptografik anahtarların güvenli ve farklı ortamlarda tutulması, c) Veriler üzerinde gerçekleştirilen tüm hareketlerin işlem kayıtlarının güvenli olarak loglanması, ç) Verilerin bulunduğu ortamlara ait güvenlik güncellemelerini 

3 farklı şartlar sağlayan bir güvenlik sağlama aracıdır. Bu kapsamda, tam disk şifrelemesiyle cihazın tümü şifrelenebilir ya da cihazda bulunan bir dosya şifrelenebilir. Bazı yazılımlar ise verilerde değişiklik yapılmasına izin vermemek için şifre koru 



In [11]:
out = ROOT / "results"
out.mkdir(exist_ok=True)
pd.DataFrame(rows).round(2).to_csv(out / "v3_summary.csv", index=False)
d.to_csv(out / "v3_per_question.csv")
print("saved")

saved
